#### ***Hyperparameter Tuning***
- ***Before training a model, we choose some values that the model does not learn automatically.***

#### ***Optuna***

- ***Optuna is a hyperparameter optimization framework, not simply a "Bayesian optimization algorithm".***

- ***I use Optuna for automated hyperparameter optimization. I define an objective function where each trial suggests parameters such as learning rate, batch size, dropout, hidden dimensions and optimizer.***

-  ***I then create and train the PyTorch model using those values and return the validation loss or another validation metric.***

-  ***Optuna runs multiple trials using a sampler such as TPE and can use pruning to stop poorly performing trials early.***

-  ***After optimization, I select the best hyperparameters, retrain the final PyTorch model appropriately, and evaluate it on an untouched test set.***

In [17]:
!pip install optuna

In [18]:
import optuna
optuna.__version__

'4.9.0'

In [19]:
## basic exmaple of optuna
import optuna
def objective(trail):
  x = trail.suggest_float("x",-10,10)
  return -(x-2)**2


study = optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=25)

print("Best value:", study.best_value)
print("Best parameters:", study.best_params)

[I 2026-08-30 02:58:22,844] A new study created in memory with name: no-name-ca20d536-8fab-41c4-8c43-5431cd7d7204
[I 2026-08-30 02:58:22,846] Trial 0 finished with value: -7.080688168875346 and parameters: {'x': 4.660956250838286}. Best is trial 0 with value: -7.080688168875346.
[I 2026-08-30 02:58:22,848] Trial 1 finished with value: -62.47116021368043 and parameters: {'x': 9.903869951718615}. Best is trial 0 with value: -7.080688168875346.
[I 2026-08-30 02:58:22,849] Trial 2 finished with value: -19.24147190632135 and parameters: {'x': 6.386510219561941}. Best is trial 0 with value: -7.080688168875346.
[I 2026-08-30 02:58:22,850] Trial 3 finished with value: -8.439294835121862 and parameters: {'x': 4.905046442851106}. Best is trial 0 with value: -7.080688168875346.
[I 2026-08-30 02:58:22,852] Trial 4 finished with value: -72.60371156477352 and parameters: {'x': -6.52078115930538}. Best is trial 0 with value: -7.080688168875346.
[I 2026-08-30 02:58:22,853] Trial 5 finished with value:

Best value: -0.01476090252346567
Best parameters: {'x': 2.1214944547025323}


In [20]:
import numpy as np
import pandas as pd

import optuna
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [21]:
## Load Dataset
df = load_breast_cancer(as_frame=True).frame
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [22]:
df.shape

(569, 31)

In [23]:
X = df.drop("target",axis=1)
y = df["target"]

In [24]:
X.shape

(569, 30)

In [25]:
# Train Test Split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,random_state=42,test_size=0.2,stratify=y
)

In [26]:
# Apply Feature Scaling
scale = StandardScaler()

X_train = scale.fit_transform(X_train)
X_test = scale.transform(X_test)

In [27]:
# Build Tensors
X_train = torch.tensor(
    X_train,dtype=torch.float32
)

X_test = torch.tensor(
    X_test,dtype=torch.float32
)

y_train = torch.tensor(
    y_train.values,dtype=torch.float32
).view(-1,1)

y_test = torch.tensor(
   y_test.values,dtype=torch.float32
).view(-1,1)

In [28]:
## Create Dataset
train_dataset = TensorDataset(
    X_train,y_train
)

test_dataset = TensorDataset(
    X_test,y_test
)

In [29]:
## Build Ourt Model

class BinaryClassifier(nn.Module):
  def __init__(self,input_dim,hidden_dim,dropout):
    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(input_dim,hidden_dim),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden_dim,1)
    )

  def forward(self,x):
    return self.network(x)

In [31]:
nn.BCEWithLogitsLoss()

BCEWithLogitsLoss()

In [33]:
from IPython.utils.py3compat import no_code
## Define Optuna Objective

def objective(trial):

  # Hyperparameter
  lr = trial.suggest_float(
      "lr", 1e-5,1e-2,log=True
  )

  hidden_dim = trial.suggest_int(
      "hidden_dim",32,256,step=32
  )

  dropout = trial.suggest_float(
      "dropout",0.0,0.5,step=0.1
  )

  batch_size = trial.suggest_categorical(
      "batch_size",[16,32,64]
  )

  optimizer_name = trial.suggest_categorical(
        "optimizer",
        ["Adam", "AdamW"]
    )

  weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        1e-2,
        log=True
    )

  ## Create DataLoader
  train_loader = DataLoader(
    train_dataset,batch_size=batch_size,shuffle=True
    )

  test_loader = DataLoader(
    test_dataset,batch_size=batch_size,shuffle=False
    )

  ## create model
  model = BinaryClassifier(
      input_dim=X_train.shape[1],
      hidden_dim=hidden_dim,
      dropout=dropout
  )

  criterion = nn.BCEWithLogitsLoss()


  if optimizer_name == "Adam":
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

  else:
    optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay
    )

  ## Training Loop
  epochs = 50
  for epoch in range(epochs):
    model.train()

    for X_batch,y_batch in train_loader:
      optimizer.zero_grad()

      outputs = model(X_batch)
      loss = criterion(outputs,y_batch)
      loss.backward()
      optimizer.step()

  ## Validation Loop
  model.eval()
  val_loss = 0.0
  with torch.no_grad():
    for X_batch,y_batch in test_loader:
      outputs = model(X_batch)

      loss = criterion(outputs,y_batch)
      val_loss+=loss.item()

  val_loss /= len(test_loader)
  return val_loss

In [34]:
study = optuna.create_study(
    direction="minimize"
)

[I 2026-08-30 03:19:41,655] A new study created in memory with name: no-name-e8605e99-dbfa-4144-af35-5b6018e5dc61


In [36]:
study.optimize(
    objective,
    n_trials=25
)

[I 2026-08-30 03:20:41,711] Trial 10 finished with value: 0.6310421675443649 and parameters: {'lr': 1.0201608408313241e-05, 'hidden_dim': 32, 'dropout': 0.2, 'batch_size': 32, 'optimizer': 'Adam', 'weight_decay': 3.011791684062976e-06}. Best is trial 4 with value: 0.07518681418150663.
[I 2026-08-30 03:20:42,845] Trial 11 finished with value: 0.23222167789936066 and parameters: {'lr': 0.00022260584238448014, 'hidden_dim': 32, 'dropout': 0.0, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 1.1470032531700366e-06}. Best is trial 4 with value: 0.07518681418150663.
[I 2026-08-30 03:20:44,461] Trial 12 finished with value: 0.09881986677646637 and parameters: {'lr': 0.0017324780808559447, 'hidden_dim': 256, 'dropout': 0.1, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 2.7252913283058523e-05}. Best is trial 4 with value: 0.07518681418150663.
[I 2026-08-30 03:20:45,130] Trial 13 finished with value: 0.2362559586763382 and parameters: {'lr': 0.000126226263427022, 'hidden_dim': 64

In [40]:
print("Best Trial:")
study.best_trial

Best Trial:


FrozenTrial(number=16, state=<TrialState.COMPLETE: 1>, values=[0.07503155060112476], datetime_start=datetime.datetime(2026, 8, 30, 3, 20, 46, 982542), datetime_complete=datetime.datetime(2026, 8, 30, 3, 20, 48, 176125), params={'lr': 0.0004977088051432664, 'hidden_dim': 224, 'dropout': 0.30000000000000004, 'batch_size': 32, 'optimizer': 'AdamW', 'weight_decay': 3.7903266881642775e-05}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'lr': FloatDistribution(high=0.01, log=True, low=1e-05, step=None), 'hidden_dim': IntDistribution(high=256, log=False, low=32, step=32), 'dropout': FloatDistribution(high=0.5, log=False, low=0.0, step=0.1), 'batch_size': CategoricalDistribution(choices=(16, 32, 64)), 'optimizer': CategoricalDistribution(choices=('Adam', 'AdamW')), 'weight_decay': FloatDistribution(high=0.01, log=True, low=1e-06, step=None)}, trial_id=16, value=None)

In [41]:
print("\nBest Validation Loss:")
study.best_value


Best Validation Loss:


0.07503155060112476

In [39]:
print("\nBest Parameters:")
study.best_params


Best Parameters:


{'lr': 0.0004977088051432664,
 'hidden_dim': 224,
 'dropout': 0.30000000000000004,
 'batch_size': 32,
 'optimizer': 'AdamW',
 'weight_decay': 3.7903266881642775e-05}

In [42]:
## Visualization Report
optuna.visualization.plot_optimization_history(
    study
)

In [43]:
# save Report to database
study = optuna.create_study(
    study_name="pytorch_tuning",
    storage="sqlite:///optuna.db",
    load_if_exists=True,
    direction="minimize"
)

[I 2026-08-30 03:25:09,286] A new study created in RDB with name: pytorch_tuning


#### ***Code Template***

In [45]:
"""def objective(trial):
    lr = trial.suggest_float(
        "lr", 1e-5, 1e-2, log=True
    )

    model = MyModel(...)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    # train
       # some code

    # validate
    some code
    return val_loss


study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=50
)

print(study.best_params)"""

'def objective(trial):\n    lr = trial.suggest_float(\n        "lr", 1e-5, 1e-2, log=True\n    )\n\n    model = MyModel(...)\n    optimizer = torch.optim.Adam(\n        model.parameters(),\n        lr=lr\n    )\n\n    # train\n       # some code \n\n    # validate\n    some code \n    return val_loss\n\n\nstudy = optuna.create_study(\n    direction="minimize"\n)\n\nstudy.optimize(\n    objective,\n    n_trials=50\n)\n\nprint(study.best_params)'